# 04 — Human Approval as a Runtime Contract

**Chapter 2 teaching lab — §2.4 + Figure 2.3**

> **Teaching outcome:** Bind human approval to immutable action payloads.

Flow: *concept → why it matters → diagram → read the implementation → expected behavior →
red-team round → playground → challenge.*

In [ ]:
# Setup: make harnesslab/ importable from any working directory, then load this lab's controls.
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "harnesslab").is_dir())
sys.path.insert(0, str(ROOT))

from harnesslab import Action, AutonomyLevel, Decision, EXECUTES, Reversibility
from datetime import datetime, timedelta, timezone
from harnesslab.approval import DecisionPacket, action_digest, approve, check_approval
from harnesslab.viz import checks, packet_card
from harnesslab.playground import approval_playground
from harnesslab.arena import NAIVE, SECURE, run_gauntlet
from harnesslab.viz import gauntlet_board, journey, show_source
from IPython.display import display

journey(4)

## Concept
An approval is not a mood ("the reviewer seemed fine with it"). It is a **decision over a specific
frozen payload, under a specific policy version, within a validity window**. At execution time the
harness recomputes the digest and refuses to run if anything differs.

The reviewer sees a **decision packet**: what will happen, why, the evidence, the expected effect, and
the rollback plan (or the explicit absence of one).

## Why this matters
Without payload binding, "approve publishing R-104" can be spent on publishing R-999. Without expiry,
last week's approval authorizes an action in a world that has since changed. Without a policy version,
a rule change silently inherits approvals granted under the old rule.

## Diagram (Figure 2.3)
```mermaid
sequenceDiagram
  participant A as Agent
  participant H as Harness
  participant R as Reviewer
  A->>H: propose publish R-104
  H->>R: DecisionPacket
  R-->>H: Approval: digest, policy 2.1, expires in 15 min
  A->>H: execute publish with payload P
  H->>H: recompute digest of P, check expiry, check policy
  alt everything matches
    H-->>A: EXECUTE
  else anything differs
    H-->>A: REFUSE
  end
```

## Read the implementation

In [ ]:
show_source(action_digest, check_approval)

## Experiment: what the reviewer sees

In [ ]:
publish = Action("publish_report", AutonomyLevel.EXECUTE_CONSEQUENTIAL, Reversibility.IRREVERSIBLE, 8, True, 10)
payload = {"report_id": "R-104", "channel": "public_site"}
packet = DecisionPacket(publish, payload, "irreversible external action",
                        ("validation passed", "sources checked"), "report visible to the public", None)
POLICY = "2.1"
now = datetime.now(timezone.utc)
ap = approve(packet, "reviewer-1", POLICY, now)
packet_card(packet)

## Expected behavior

In [ ]:
assert check_approval(ap, "publish_report", payload, POLICY, now) == (True, "valid")
assert check_approval(ap, "publish_report", payload, POLICY, now + timedelta(minutes=14)) == (True, "valid")
print("The exact approved action executes inside its window.")

## Every way to misuse an approval

In [ ]:
attacks = {
    "tampered payload":         ("publish_report", {**payload, "report_id": "R-999"}, POLICY, now),
    "added field":              ("publish_report", {**payload, "cc": "press@example.com"}, POLICY, now),
    "replayed on other action": ("delete_report", payload, POLICY, now),
    "expired":                  ("publish_report", payload, POLICY, now + timedelta(minutes=20)),
    "policy changed":           ("publish_report", payload, "2.2", now),
}
results = {label: check_approval(ap, *args) for label, args in attacks.items()}
assert not any(ok for ok, _ in results.values())
checks([(label, ok, why) for label, (ok, why) in results.items()], title="Misuse attempts")

## Red-team round (§2.4)
The same attacks run against two defense sets: **HarnessLab** (what this lab built) and
**Shortcuts** (the patterns the chapter warns about). Every attack must be blocked by HarnessLab.
The Shortcuts defense accepts any approval that has a reviewer's name on it.

In [ ]:
secure, shortcuts = run_gauntlet(SECURE, "2.4"), run_gauntlet(NAIVE, "2.4")
assert all(r.blocked for r in secure), [r.attack.name for r in secure if not r.blocked]
gauntlet_board(secure, shortcuts, title="Gauntlet · §2.4")

## Playground
Edit the payload JSON, switch the action, move time forward, or bump the policy version.

In [ ]:
approval_playground(ap, now, payload)

## Challenge
1. Require **two distinct approvers** when `blast_radius >= 8`. The agent's requester identity must not
   count as an approver.
2. Add `max_uses` so one approval cannot execute the same payload twice.
3. The packet's `rollback_plan` is `None`. Should the harness refuse to even *request* approval for an
   irreversible action with no rollback plan, or is "NONE" useful information for the reviewer?